# 10.6 Indexed collections of tables and columns

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

In some circumstances, it is convenient to declare an indexed collection of tables, or
to define an indexed collection of `data` columns within a `table`. This section explains how
indexing of these kinds can be specified within the `table` declaration.

To illustrate indexed collections of tables, we present a script ([Chapter 13](#missing) <!--- xTODO missing reference --->) that automatically
solves a series of scenarios stored separately. To illustrate indexed collections
of columns, we show how a two-dimensional spreadsheet `table` can be read.

All of our examples of these features make use of AMPL's character-string expressions
to generate names for series of files, tables, or columns. For more on string expressions,
see Sections 13.7 and A.4.2.

Indexed collections of tables
AMPL `table` declarations can be indexed in much the same way as sets, parameters,
and other `model` components. An optional {indexing-expr} follows the `table`-name:

```ampl
table table-name {indexing-expr} opt string-list opt : ...
```

**Figure 10-5:** Access database with tables of sensitivity analysis.

One `table` is defined for each member of the set specified by the indexing-expr. Individual
tables in this collection are denoted in the usual way, by appending a bracketed subscript
or subscripts to the `table`-name.

As an example, the following declaration defines a collection of AMPL tables indexed
over the set of foods in diet.mod, each `table` corresponding to a different database `table`
in the Access file DietSens.mdb:

```ampl
table DietSens {j in FOOD}
   OUT "ODBC" "DietSens.mdb" ("Sens" & j):
      [Food], f_min, Buy, f_max;
```

Following the rules for the standard ODBC `table` handler, the Access `table` names are
given by the third item in the string-list, the string expression ("Sens" & j). Thus
the AMPL `table` DietSens["BEEF"] is associated with the Access `table` SensBEEF,
the AMPL `table` DietSens["CHK"] is associated with the Access `table` SensCHK, and
so forth. The following AMPL script uses these tables to record the optimal diet when
there is a two-for-the-price-of-one sale on each of the foods:

```ampl
for {j in FOOD} {
   let cost[j] := cost[j] / 2;
   solve;
   write table DietSens[j];
   let cost[j] := cost[j] * 2;
}
```

**Figure 10-6:** Alternate Access `table` for sensitivity analysis.

For the `data` in diet2a.dat, the set FOOD has eight members, so eight tables are written
in the Access database, as seen in Figure 10-5. If instead the `table` declaration were
to give a string expression for the second string in the string-list, which specifies the
Access filename:

```ampl
table DietSens {j in FOOD}
   OUT "ODBC" ("DietSens" & j & ".mdb"):
      [Food], f_min, Buy, f_max;
```

then AMPL would write eight different Access database files, named
DietSensBEEF.mdb, DietSensCHK.mdb, and so forth, each containing a single
`table` named (by default) DietSens. (These files must have been created before the
`write table` commands are executed.)
A string expression can be used in a similar way to make every member of an indexed
collection of AMPL tables correspond to the same Access `table`, but with a different datacol
-name for the optimal amounts:

```ampl
table DietSens {j in FOOD} "ODBC" "DietSens.mdb":
   [Food], Buy ˜ ("Buy" & j);
```

Then running the script shown above will result in the Access `table` of Figure 10-6. The
AMPL tables in this case were deliberately left with the default read/write status, INOUT.
Had the read/write status been specified as OUT, then each `write table` would have
overwritten the columns created by the previous one.
**Figure 10-7:** Two-dimensional AMPL `table` in Excel.

Indexed collections of `data` columns
Because there is a natural correspondence between `data` columns of a relational `table`
and indexed collections of entities in an AMPL `model`, each `data`-spec in a `table` declaration
normally refers to a different AMPL parameter, variable, or expression. Occasionally
the values for one AMPL entity are split among multiple `data` columns, however.
Such a case can be handled by defining a collection of `data` columns, one for each member
of a specified indexing set.

The most common use of this feature is to read or write two-dimensional tables. For
example, the `data` for the parameter

```ampl
param amt {NUTR,FOOD} >= 0;
```

from diet.mod might be represented in an Excel spreadsheet as a `table` with nutrients
labeling the rows and foods the columns (Figure 10-7). To read this `table` using AMPL's
external database features, we must regard it as having one key column, under the heading
NUTR, and `data` columns headed by the names of individual foods. Thus we require a
`table` declaration whose key-spec is one-dimensional and whose `data`-specs are indexed
over the AMPL set FOOD:

```ampl
table dietAmts IN "ODBC" "Diet2D.xls":
   [i ˜ NUTR], {j in FOOD} <amt[i,j] ˜ (j)>;
```

The key-spec [i ˜ NUTR] associates the first `table` column with the set NUTR. The
`data`-spec {j in FOOD} <...> causes AMPL to generate an individual `data`-spec for each
member of set FOOD. Specifically, for each `j` in FOOD, AMPL generates the `data`-spec
`amt[i,j] ˜ (j)`, where (j) is the AMPL string expression for the heading of the
external `table` column for food j, and amt[i,j] denotes the parameter to which the values
in that column are to be written. (According to the convention used here and in other
AMPL declarations and commands, the parentheses around (j) cause it to be interpreted
as an expression for a string; without the parentheses it would denote a column name consisting
of the single character j.)
A similar approach works for writing two-dimensional tables to spreadsheets. As an
example, after steelT.mod is solved, the results could be written to a spreadsheet
using the following `table` declaration:

```ampl
table Results1 OUT "ODBC" "steel1out.xls":
   {p in PROD} -> [Product],
      Inv[p,0] ˜ Inv0,
      {t in 1..T} < Make[p,t] ˜ ('Make' & t),
                    Sell[p,t] ˜ ('Sell' & t),
                    Inv[p,t] ˜ ('Inv' & t) >;
```

or, equivalently, using `display`-style indexing:

```ampl
table Results2 OUT "ODBC" "steel2out.xls":
   [Product],
      {p in PROD} Inv[p,0] ˜ Inv0,
      {t in 1..T} < {p in PROD} (Make[p,t] ˜ ('Make' & t),
                                 Sell[p,t] ˜ ('Sell' & t),
                                 Inv[p,t] ˜ ('Inv' & t) ) >;
```

The key column labels the rows with product names. The `data` columns include one for
the initial inventories, and then three representing production, sales, and inventories,
respectively, for each period, as in Figure 10-8. Conceptually, there is a symmetry
between the row and column indexing of a two-dimensional `table`. But because the tables
in these examples are being treated as relational tables, the `table` declaration must treat
the row indexing and the column indexing in different ways. As a result, the expressions
describing row indexing are substantially different from those describing column indexing.

As these examples suggest, the general form for specifying an indexed collection of
`table` columns is

```ampl
{indexing-expr} < data-spec, data-spec, data-spec, ... >
```

where each `data`-spec has any of the forms previously given. For each member of the set
specified by the indexing-expr, AMPL generates one copy of each `data`-spec within the
angle brackets <...>. The indexing-expr also defines one or more dummy indices that run
over the index set; these indices are used in expressions within the `data`-specs, and also
appear in string expressions that give the names of columns in the external database.